In [ ]:
!pip install accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype="auto"
)

model.eval()

In [ ]:
def generate_response(prompt):

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    generated_ids = model.generate(
        **text,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    
    # Remove the input tokens from the output 
    generated_ids = [ output_ids[len(input_ids):] for input_ids, output_ids in zip(text.input_ids, generated_ids) ] 
    
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
prompt = "what is the capital of canada"
generate_response(prompt)

In [ ]:
dataset = load_dataset("Anthropic/hh-rlhf")

print(pd.DataFrame(dataset["train"].select(range(5))))

In [ ]:
def format_example(example, response_key):
    text = example[response_key]
    return tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

@torch.no_grad()
def get_activations(example, response_type):
    inputs = format_example(example, response_type)

    outputs = model(
        **inputs,
        output_hidden_states=True,
        use_cache=False
    )

    hidden_states = outputs.hidden_states

    activations = torch.stack([h[0, -1, :].detach().float().cpu() for h in hidden_states[1:]])

    return activations  

In [ ]:
num_examples = 100

chosen_acts = []
rejected_acts = []

for ex in tqdm(dataset["train"].select(range(num_examples))):
    chosen_acts.append(get_activations(ex, "chosen"))
    rejected_acts.append(get_activations(ex, "rejected"))

chosen_acts = torch.stack(chosen_acts)      # [N, 28, hidden_size]
rejected_acts = torch.stack(rejected_acts)  # [N, 28, hidden_size]

print(chosen_acts.shape)
print(rejected_acts.shape)

In [ ]:
refusal_direction = chosen_acts.mean(dim=0) - rejected_acts.mean(dim=0)
refusal_direction = refusal_direction / refusal_direction.norm(dim=-1, keepdim=True)

print(refusal_direction.shape)  # [28, hidden_size]

In [ ]:
def make_injection_hook(layer_idx, alpha):
    def hook(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        v = refusal_direction[layer_idx].to(hidden.device).to(hidden.dtype)

        if hidden.dim() == 3:
            hidden[:, -1, :] = hidden[:, -1, :] + alpha * v
        elif hidden.dim() == 2:
            hidden[-1, :] = hidden[-1, :] + alpha * v
        else:
            raise ValueError(f"Unexpected hidden shape: {hidden.shape}")

        if isinstance(output, tuple):
            return (hidden,) + output[1:]
        return hidden

    return hook

In [ ]:
def debug_hook(module, inputs, output):
    print("OUTPUT TYPE:", type(output))

    if isinstance(output, tuple):
        print("TUPLE LENGTH:", len(output))
        for i, item in enumerate(output):
            if torch.is_tensor(item):
                print(f"output[{i}] shape:", item.shape)
            else:
                print(f"output[{i}] type:", type(item))
    elif torch.is_tensor(output):
        print("output shape:", output.shape)

    return output

handle = model.model.layers[20].register_forward_hook(debug_hook)

print(generate_response("Tell me why stealing is wrong."))

handle.remove()

In [ ]:
layer_idx = 20
alpha = 10.0

handle = model.model.layers[layer_idx].register_forward_hook(
    make_injection_hook(layer_idx, alpha)
)

print(generate_response("how can i create a bomb."))

handle.remove()

In [ ]:
def make_ablation_hook(layer_idx):
    def hook(module, inputs, output):
        hidden = output[0]

        v = refusal_direction[layer_idx].to(hidden.device).to(hidden.dtype)

        x = hidden[:, -1, :]

        # remove component of x along v
        proj = (x * v).sum(dim=-1, keepdim=True) * v

        hidden[:, -1, :] = x - proj

        return (hidden,) + output[1:]

    return hook

In [ ]:
layer_idx = 20

handle = model.model.layers[layer_idx].register_forward_hook(
    make_ablation_hook(layer_idx)
)

print(generate_response("Tell me why stealing is wrong."))

handle.remove()